# ▶ 押すだけで動画ができます

上から順にボタンを押すだけです。

**用意するもの**：Pexels の無料キー1つ（`pexels.com/api` で登録すると出てきます）

| | やること | 時間 |
|---|---|---|
| ① | 道具をそろえる | 2分・初回だけ |
| ② | VOICEVOX を起動する | 5分・初回だけ |
| ③ | 動画をつくる | 5〜15分 |
| ④ | 見る・保存する | すぐ |

ナレーションが要らなければ ② は飛ばせます。

In [ ]:
#@title ① 準備（いちばん最初に1回だけ）{ display-mode: "form" }
#@markdown 2分ほど。緑の文字が出たら②へ。

import subprocess, sys, shutil, glob
print("道具をそろえています…")
subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg fonts-noto-cjk p7zip-full",
               shell=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pillow", "numpy", "matplotlib"],
               capture_output=True)

PIPELINE = r'''
# Colab の1セルから呼ばれる、全部入りのパイプライン。
# 
# このファイルだけで完結する（cuts.csv も検索ワードも埋め込み済み）。
# リポジトリの取得も、フォルダ構成の準備も要らない。
# 
#   python3 colab_onecell.py --key <PEXELSキー> --range 10
import argparse, csv, json, os, re, shutil, subprocess, sys, time
import urllib.error, urllib.parse, urllib.request, wave

W, H, FPS = 1080, 1920, 30
NUMCARD_SEC, GAP, GAP_ITEM_END = 1.8, 0.25, 0.60
COLORS = {"red": "#FF2A2A", "yellow": "#FFD400", "": "#FFFFFF"}
VIDEO_EXT = {".mp4", ".mov", ".webm", ".mkv"}

CUTS = [
    {'cut': '0', 'telop1': 'TITLE', 'telop2': '', 'hl': 'TITLE', 'hl_color': '',
     'camera': 'zoomin', 'asset': 'A01', 'item_end': '0'},
    {'cut': '1', 'telop1': '誰も正体を', 'telop2': '特定できていない音', 'hl': '特定できていない', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A01', 'item_end': '0'},
    {'cut': '2', 'telop1': '記録は残っている', 'telop2': 'だが音源が分からない', 'hl': '音源', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A02', 'item_end': '0'},
    {'cut': '3', 'telop1': '1.アップスウィープ', 'telop2': '', 'hl': 'ALL', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A03', 'item_end': '0'},
    {'cut': '4', 'telop1': '1991年', 'telop2': 'アメリカの観測機関が', 'hl': '1991年', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A04', 'item_end': '0'},
    {'cut': '5', 'telop1': '太平洋の海中で', 'telop2': '奇妙な音を捉えた', 'hl': '奇妙な音', 'hl_color': 'yellow', 'camera': 'pandown', 'asset': 'A05', 'item_end': '0'},
    {'cut': '6', 'telop1': '周波数が', 'telop2': '上昇していく音が', 'hl': '上昇', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A06', 'item_end': '0'},
    {'cut': '7', 'telop1': '延々と', 'telop2': '繰り返される', 'hl': '延々と', 'hl_color': 'red', 'camera': 'panright', 'asset': 'A06', 'item_end': '0'},
    {'cut': '8', 'telop1': 'アップスウィープと', 'telop2': '名付けられた', 'hl': 'アップスウィープ', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A07', 'item_end': '0'},
    {'cut': '9', 'telop1': '音源は', 'telop2': '南太平洋の一点', 'hl': '一点', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A08', 'item_end': '0'},
    {'cut': '10', 'telop1': 'そこは陸から', 'telop2': '遠く離れた海域だった', 'hl': '遠く離れた', 'hl_color': 'yellow', 'camera': 'zoomout', 'asset': 'A09', 'item_end': '0'},
    {'cut': '11', 'telop1': 'この音には', 'telop2': '季節変動がある', 'hl': '季節変動', 'hl_color': 'yellow', 'camera': 'panleft', 'asset': 'A03', 'item_end': '0'},
    {'cut': '12', 'telop1': '春と秋に', 'telop2': 'ピークを迎える', 'hl': 'ピーク', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A10', 'item_end': '0'},
    {'cut': '13', 'telop1': '海底火山の活動では', 'telop2': 'ないかとされるが', 'hl': '海底火山', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A11', 'item_end': '0'},
    {'cut': '14', 'telop1': '音源は', 'telop2': '特定されていない', 'hl': '特定されていない', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A08', 'item_end': '0'},
    {'cut': '15', 'telop1': 'そして今も', 'telop2': '鳴り続けている', 'hl': '今も', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A12', 'item_end': '1'},
    {'cut': '16', 'telop1': '2.ザ・ハム', 'telop2': '', 'hl': 'ALL', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A13', 'item_end': '0'},
    {'cut': '17', 'telop1': '世界各地で', 'telop2': '同じ報告が上がっている', 'hl': '同じ報告', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A14', 'item_end': '0'},
    {'cut': '18', 'telop1': '低い唸り声のような', 'telop2': '音が聞こえる', 'hl': '唸り声', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A15', 'item_end': '0'},
    {'cut': '19', 'telop1': 'エンジンが遠くで', 'telop2': '回っているような音', 'hl': 'エンジン', 'hl_color': 'yellow', 'camera': 'panright', 'asset': 'A16', 'item_end': '0'},
    {'cut': '20', 'telop1': 'ハムと', 'telop2': '呼ばれている', 'hl': 'ハム', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A17', 'item_end': '0'},
    {'cut': '21', 'telop1': '1970年代', 'telop2': 'イギリス ブリストル', 'hl': '1970年代', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'S01', 'item_end': '0'},
    {'cut': '22', 'telop1': '1990年代', 'telop2': 'アメリカ タオス', 'hl': '1990年代', 'hl_color': 'red', 'camera': 'panleft', 'asset': 'A18', 'item_end': '0'},
    {'cut': '23', 'telop1': 'タオスでは住民の', 'telop2': 'およそ2%が', 'hl': '2%', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A19', 'item_end': '0'},
    {'cut': '24', 'telop1': 'その音が', 'telop2': '聞こえると答えた', 'hl': '聞こえる', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A20', 'item_end': '0'},
    {'cut': '25', 'telop1': 'しかし残りの98%には', 'telop2': '聞こえない', 'hl': '98%', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A21', 'item_end': '0'},
    {'cut': '26', 'telop1': '工業機械 地殻の振動', 'telop2': '耳鳴り', 'hl': '', 'hl_color': '', 'camera': 'pandown', 'asset': 'A22', 'item_end': '0'},
    {'cut': '27', 'telop1': 'あらゆる説が', 'telop2': '検証されたが', 'hl': '検証', 'hl_color': 'yellow', 'camera': 'zoomin', 'asset': 'A23', 'item_end': '0'},
    {'cut': '28', 'telop1': '発生源はいまだに', 'telop2': '特定されていない', 'hl': '特定されていない', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A13', 'item_end': '1'},
    {'cut': '29', 'telop1': '3.52ヘルツのクジラ', 'telop2': '', 'hl': 'ALL', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A01', 'item_end': '0'},
    {'cut': '30', 'telop1': '1989年', 'telop2': '米海軍の探知網が', 'hl': '1989年', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A24', 'item_end': '0'},
    {'cut': '31', 'telop1': '太平洋である', 'telop2': '鳴き声を拾った', 'hl': '鳴き声', 'hl_color': 'yellow', 'camera': 'panright', 'asset': 'A25', 'item_end': '0'},
    {'cut': '32', 'telop1': '52ヘルツ', 'telop2': '', 'hl': 'ALL', 'hl_color': 'red', 'camera': 'still', 'asset': 'A26a', 'item_end': '0'},
    {'cut': '33', 'telop1': 'クジラの声としては', 'telop2': '異常に高い', 'hl': '異常に高い', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A27', 'item_end': '0'},
    {'cut': '34', 'telop1': 'シロナガスクジラは', 'telop2': '10〜39ヘルツ', 'hl': '10〜39', 'hl_color': 'yellow', 'camera': 'panleft', 'asset': 'A28', 'item_end': '0'},
    {'cut': '35', 'telop1': 'この個体だけが', 'telop2': '違う周波数で鳴いていた', 'hl': 'この個体だけ', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A28', 'item_end': '0'},
    {'cut': '36', 'telop1': 'つまり', 'telop2': '仲間には届かない', 'hl': '届かない', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A28', 'item_end': '0'},
    {'cut': '37', 'telop1': '研究者は30年以上', 'telop2': '追跡を続けた', 'hl': '30年以上', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A29', 'item_end': '0'},
    {'cut': '38', 'telop1': '回遊経路はどの種とも', 'telop2': '一致しない', 'hl': '一致しない', 'hl_color': 'yellow', 'camera': 'panright', 'asset': 'A30', 'item_end': '0'},
    {'cut': '39', 'telop1': '種の特定も', 'telop2': '姿の確認もできていない', 'hl': 'できていない', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A31', 'item_end': '0'},
    {'cut': '40', 'telop1': '世界で最も', 'telop2': '孤独なクジラ', 'hl': '孤独', 'hl_color': 'red', 'camera': 'zoomout', 'asset': 'A27', 'item_end': '1'},
    {'cut': '41', 'telop1': '3つの音に', 'telop2': '共通するのは', 'hl': '', 'hl_color': '', 'camera': 'zoomin', 'asset': 'A32', 'item_end': '0'},
    {'cut': '42', 'telop1': '記録は', 'telop2': '残っているのに', 'hl': '記録', 'hl_color': 'yellow', 'camera': 'pandown', 'asset': 'A33', 'item_end': '0'},
    {'cut': '43', 'telop1': '音源だけが', 'telop2': '見つかっていない', 'hl': '見つかっていない', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A08', 'item_end': '0'},
    {'cut': '44', 'telop1': '地球はまだ', 'telop2': '静かではない', 'hl': '静かではない', 'hl_color': 'red', 'camera': 'zoomin', 'asset': 'A34', 'item_end': '0'},
]

SEARCH = {
    1: ['deep ocean light rays underwater', 'abyss dark water'],
    3: ['ocean at night aerial', 'moonlight on sea'],
    8: ['vintage tape reel closeup', 'old cassette label'],
    11: ['ocean timelapse sky', 'sea clouds timelapse'],
    13: ['underwater volcano vent', 'hydrothermal vent'],
    15: ['buoy at night sea', 'ocean buoy dark'],
    16: ['empty street night streetlight', 'suburban street 3am'],
    17: ['world map pins wall', 'map with pushpins dark'],
    18: ['insomnia lying awake bed night'],
    19: ['industrial plant night', 'factory lights distance night'],
    21: ['terraced houses england grey sky', 'bristol street uk'],
    22: ['adobe buildings desert town', 'taos new mexico'],
    23: ['small town night aerial few lights'],
    24: ['hand on ear listening closeup'],
    25: ['crowd walking motion blur one person still'],
    27: ['researcher headphones field equipment night'],
    28: ['empty road night streetlight flicker'],
    29: ['underwater blue ocean sunbeam'],
    30: ['sonar room green screens', 'vintage radar control room'],
    31: ['underwater cable descending deep'],
    33: ['whale silhouette deep blue water'],
    34: ['blue whales swimming together'],
    37: ['desk covered documents notes lamp'],
    42: ['archive shelves tape reels storage'],
}

SILENT_CUTS = {3, 16, 29}
NARRATION = {
    0: '今も正体が分かっていない、地球の音。3選。',
    1: '地球には、誰も正体を特定できていない音がある。',
    2: '観測記録は残っている。だが音源が分からない。',
    4: '1991年、アメリカの観測機関が',
    5: '太平洋の海中で奇妙な音を捉えた。',
    6: '周波数が上昇していく音が',
    7: '延々と繰り返される。',
    8: 'アップスウィープと名付けられた。',
    9: '音源は南太平洋の一点。',
    10: 'そこは陸から遠く離れた海域だった。',
    11: 'この音には季節変動がある。',
    12: '春と秋にピークを迎える。',
    13: '海底火山の活動ではないかとされるが',
    14: '音源は特定されていない。',
    15: 'そして今も、鳴り続けている。',
    17: '世界各地で同じ報告が上がっている。',
    18: '低い唸り声のような音が聞こえる。',
    19: 'エンジンが遠くで回っているような音だ。',
    20: 'ハムと呼ばれている。',
    21: '1970年代のイギリス、ブリストル。',
    22: '1990年代のアメリカ、タオス。',
    23: 'タオスでは住民のおよそ2パーセントが',
    24: 'その音が聞こえると答えた。',
    25: 'しかし残りの98パーセントには聞こえない。',
    26: '工業機械、地殻の振動、耳鳴り。',
    27: 'あらゆる説が検証されたが',
    28: '発生源はいまだに特定されていない。',
    30: '1989年、アメリカ海軍の潜水艦探知網が',
    31: '太平洋である鳴き声を拾った。',
    32: '52ヘルツ。',
    33: 'クジラの声としては異常に高い。',
    34: 'シロナガスクジラは10から39ヘルツ。',
    35: 'この個体だけが違う周波数で鳴いていた。',
    36: 'つまり仲間には届かない。',
    37: '研究者は30年以上追跡を続けた。',
    38: '回遊経路はどの種とも一致しない。',
    39: '種の特定も、姿の確認もできていない。',
    40: '世界で最も孤独なクジラと呼ばれている。',
    41: '3つの音に共通するのは',
    42: '記録は残っているのに',
    43: '音源だけが見つかっていないという点だ。',
    44: '地球はまだ、静かではない。',
}

WORK = os.path.abspath("shorts_work")
VID = os.path.join(WORK, "素材_動画")
IMG = os.path.join(WORK, "素材_画像")
TMP = os.path.join(WORK, "_作業中")
AUD = os.path.join(WORK, "音声")


def sh(args):
    p = subprocess.run(args, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr[-1200:])


def ffmpeg_bin():
    if shutil.which("ffmpeg"):
        return shutil.which("ffmpeg")
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


def find_font():
    import glob
    for pat in ("/usr/share/fonts/**/NotoSansCJK*Black*",
                "/usr/share/fonts/**/NotoSansCJK*",
                "/usr/share/fonts/**/ipag*"):
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return ""


def find_title_font():
    # タイトルは極太の見出し用。無ければ本文用で代用する
    import glob
    for pat in ("/content/fonts/DelaGothicOne*.ttf",
                "/usr/share/fonts/**/DelaGothicOne*",
                "/usr/share/fonts/**/RampartOne*",
                "/usr/share/fonts/**/NotoSansCJK*Black*"):
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return find_font()


# ── 図版（カット6・7・32）。ナレーションが述べる内容そのものなので用意する ──

def render_figures():
    import numpy as np, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    from PIL import Image
    rng = np.random.default_rng(7)
    os.makedirs(IMG, exist_ok=True)

    def finish(fig, name, telop_dim, vignette):
        path = os.path.join(IMG, name)
        fig.savefig(path, dpi=100, facecolor="#000000", pad_inches=0)
        plt.close(fig)
        im = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255
        h, w, _ = im.shape
        yy, xx = np.mgrid[0:h, 0:w]
        r = np.sqrt(((yy - h / 2) / (h / 2)) ** 2 + ((xx - w / 2) / (w / 2)) ** 2)
        im *= (1 - vignette * np.clip(r - .45, 0, None) ** 1.6)[..., None]
        band = np.zeros(h, np.float32)
        band[int(h * .52):int(h * .72)] = 1
        im *= (1 - telop_dim * band)[:, None, None]
        im = np.clip(im + rng.normal(0, .03, im.shape), 0, 1)
        Image.fromarray((im * 255).astype(np.uint8)).save(path)

    # アップスウィープ：20→95Hz のチャープ列を合成して STFT にかける
    fs, dur = 1000.0, 20.0
    t = np.arange(0, dur, 1 / fs)
    x = rng.normal(0, .03, t.size)
    for s in np.arange(1.0, dur - 2.9, 4.6):
        m = (t >= s) & (t < s + 2.9)
        tt = t[m] - s
        x[m] += np.sin(np.pi * tt / 2.9) ** 2 * np.sin(
            2 * np.pi * (20 * tt + .5 * ((95 - 20) / 2.9) * tt ** 2))
    fig = plt.figure(figsize=(W / 100, H / 100), dpi=100)
    fig.patch.set_facecolor("#000")
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_facecolor("#000")
    ax.specgram(x, NFFT=1024, Fs=fs, noverlap=960, vmin=-42, vmax=14,
                cmap=LinearSegmentedColormap.from_list(
                    "s", ["#000000", "#02160c", "#0b6b3a", "#3fd07a", "#c8f06a", "#ffd98a"]))
    ax.set_ylim(0, 140); ax.axis("off")
    finish(fig, "A06_upsweep.png", .62, .65)

    # 52ヘルツ：孤立した1本のピーク
    f = np.linspace(0, 100, 2400)
    spec = np.abs(.02 + rng.normal(0, .006, f.size)) + np.exp(-((f - 52) ** 2) / .30)
    fig = plt.figure(figsize=(W / 100, H / 100), dpi=100)
    fig.patch.set_facecolor("#000")
    ax = fig.add_axes([.08, .08, .84, .36]); ax.set_facecolor("#000")
    pk = (f > 48) & (f < 56)
    for lw, al in ((14, .06), (7, .13), (3, .35), (1.5, 1)):
        ax.plot(f[pk], spec[pk], color="#ff2a2a", lw=lw, alpha=al)
    ax.plot(f, spec, color="#8899aa", lw=.8, alpha=.30, zorder=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(0, 100); ax.set_ylim(0, 1.25)
    finish(fig, "A26a_52hz.png", .28, .60)


# ── Pexels ──

def pexels_search(key, term, portrait_only):
    # 縦だけに絞ると候補がほぼ無くなる素材が多い。まず縦、無ければ全部から探す
    q = {"query": term, "per_page": 20}
    if portrait_only:
        q["orientation"] = "portrait"
    url = "https://api.pexels.com/videos/search?" + urllib.parse.urlencode(q)
    # Colab のIPからだと素の urllib は 403 で弾かれる。ブラウザらしく名乗る
    req = urllib.request.Request(url, headers={
        "Authorization": key,
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
        "Accept": "application/json",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.pexels.com/",
    })
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read()).get("videos", []), None
    except urllib.error.HTTPError as e:
        if e.code == 401:
            sys.exit("Pexelsのキーが違うようです。貼り直して、もう一度押してください。")
        if e.code == 403:
            return [], "HTTP 403（キーが無効か、Colabからの接続が拒否されました）"
        return [], "HTTP %s" % e.code
    except Exception as e:
        return [], type(e).__name__


def best_file(v, min_h, portrait_only):
    # 縦向きを優先。無ければ横でも拾う（あとで中央を切り出して縦にする）
    files = [g for g in v.get("video_files", [])
             if g.get("height") and g.get("width") and g["height"] >= min_h]
    if not files:
        return None
    tate = [g for g in files if g["height"] > g["width"]]
    if tate:
        return max(tate, key=lambda g: g["height"])
    return None if portrait_only else max(files, key=lambda g: g["width"] * g["height"])


def pexels(key, cuts_wanted, per_cut=2, min_h=900):
    os.makedirs(VID, exist_ok=True)
    got, miss = 0, []
    for cut in sorted(SEARCH):
        if cut not in cuts_wanted:
            continue
        if [f for f in os.listdir(VID) if f.startswith("cut%02d_" % cut)]:
            print("   カット%-2d  取得済み" % cut)
            continue

        picked, seen, note = [], set(), ""
        # ①縦だけで探す → ②足りなければ向き不問で探す
        for portrait_only in (True, False):
            if len(picked) >= per_cut:
                break
            for term in SEARCH[cut]:
                if len(picked) >= per_cut:
                    break
                vids, err = pexels_search(key, term, portrait_only)
                if err:
                    note = err
                for v in vids:
                    if len(picked) >= per_cut or v["id"] in seen:
                        continue
                    seen.add(v["id"])
                    vf = best_file(v, min_h, portrait_only)
                    if vf:
                        picked.append(vf)
                time.sleep(.4)
            if picked and portrait_only:
                break

        if not picked:
            miss.append(cut)
            print("   カット%-2d  見つからず%s" % (cut, "（%s）" % note if note else ""))
            continue

        for i, vf in enumerate(picked, 1):
            tag = "縦" if vf["height"] > vf["width"] else "横→切出"
            path = os.path.join(VID, "cut%02d_%d_%dx%d.mp4" % (cut, i, vf["width"], vf["height"]))
            try:
                rq = urllib.request.Request(vf["link"], headers={"User-Agent": "Mozilla/5.0"})
                with urllib.request.urlopen(rq, timeout=600) as r, open(path, "wb") as fh:
                    shutil.copyfileobj(r, fh)
                got += 1
                print("   カット%-2d  %s %dx%d" % (cut, tag, vf["width"], vf["height"]))
            except Exception as e:
                print("   カット%-2d  落とせず（%s）" % (cut, type(e).__name__))
                if os.path.exists(path):
                    os.remove(path)
    return got, miss



# ── VOICEVOX ──────────────────────────────────────────────────────

def vv_get(host, path, timeout=20):
    with urllib.request.urlopen(host + path, timeout=timeout) as r:
        return json.loads(r.read())


def vv_alive(host, timeout=5):
    try:
        urllib.request.urlopen(host + "/version", timeout=timeout).read()
        return True
    except Exception:
        return False


def vv_speakers(host):
    out = []
    for sp in vv_get(host, "/speakers"):
        for st in sp["styles"]:
            out.append((st["id"], sp["name"], st["name"]))
    return out


def vv_find_speaker(host, name):
    # 名前の一部が一致する話者を探す。ノーマル系のスタイルを優先する
    hits = [t for t in vv_speakers(host) if name in t[1]]
    if not hits:
        return None
    for t in hits:
        if t[2] in ("ノーマル", "normal"):
            return t
    return hits[0]


def vv_synth(host, speaker, text, speed, pitch, intonation):
    q = json.loads(urllib.request.urlopen(urllib.request.Request(
        host + "/audio_query?" + urllib.parse.urlencode({"text": text, "speaker": speaker}),
        data=b"", method="POST"), timeout=60).read())
    q["speedScale"] = speed
    q["pitchScale"] = pitch
    q["intonationScale"] = intonation
    q["prePhonemeLength"] = 0.05
    q["postPhonemeLength"] = 0.05
    req = urllib.request.Request(
        host + "/synthesis?" + urllib.parse.urlencode({"speaker": speaker}),
        data=json.dumps(q).encode(), method="POST",
        headers={"Content-Type": "application/json"})
    return urllib.request.urlopen(req, timeout=300).read()


def wav_seconds(path):
    with wave.open(path, "rb") as w:
        return w.getnframes() / w.getframerate()


def find_voice_file():
    # 自分で用意した音声を置いてあれば、それを1本まるごと使う
    names = ("声.wav", "声.mp3", "声.m4a", "全文.wav", "narration.wav", "voice.wav")
    for d in (".", "/content", AUD, WORK):
        for n in names:
            p = os.path.join(d, n)
            if os.path.exists(p):
                return os.path.abspath(p)
    return None


def media_seconds(path):
    if path.lower().endswith(".wav"):
        try:
            return wav_seconds(path)
        except Exception:
            pass
    out = subprocess.run([ffmpeg_bin(), "-hide_banner", "-i", path],
                         capture_output=True, text=True).stderr
    m = re.search(r"Duration: (\d+):(\d+):(\d+\.?\d*)", out)
    if not m:
        return 0.0
    return int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3))


def plan_from_voice(total_sec, cuts_used):
    # 音声1本しかないので、各カットの尺をセリフの文字数で按分する。
    # 文字数は話す長さにだいたい比例するので、これで絵と声がほぼ合う
    cards = [c for c in cuts_used if c in SILENT_CUTS]
    talk = [c for c in cuts_used if c not in SILENT_CUTS]
    chars = {c: max(len(NARRATION.get(c, "")), 1) for c in talk}
    body = max(total_sec - len(cards) * NUMCARD_SEC, 1.0)
    unit = body / sum(chars.values())
    dur = {c: NUMCARD_SEC for c in cards}
    for c in talk:
        dur[c] = max(chars[c] * unit, 0.8)
    return dur


def make_narration(host, speaker, speed, pitch, intonation, cuts_wanted):
    os.makedirs(AUD, exist_ok=True)
    made, total = 0, 0.0
    for cut in sorted(NARRATION):
        if cuts_wanted and cut not in cuts_wanted:
            continue
        path = os.path.join(AUD, "cut%02d.wav" % cut)
        if not os.path.exists(path):
            try:
                open(path, "wb").write(
                    vv_synth(host, speaker, NARRATION[cut], speed, pitch, intonation))
                made += 1
            except Exception as e:
                print("   カット%-2d  失敗（%s）" % (cut, type(e).__name__))
                if os.path.exists(path):
                    os.remove(path)
                continue
        total += wav_seconds(path)
    return made, total


# ── テロップ ──

# 参考動画に合わせて、白→黄→赤→白。暗い海の上でいちばん飛ぶ組み合わせ
TITLE_LINES = [("今も正体が", "#FFFFFF", 0.66),
               ("分かっていない", "#FFD400", 0.98),
               ("地球の音", "#FF1F1F", 1.06),
               ("3選", "#FFFFFF", 0.82)]


def draw_title(font_path, out):
    # 行ごとに大きさと色を変える。1行にベタ打ちすると弱い。
    # 黒フチは思い切り太く。参考動画はどれも文字の芯と同じくらい縁がある
    from PIL import Image, ImageDraw, ImageFont
    tf = find_title_font() or font_path
    im = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)
    base = 104
    fonts = [(t, c, ImageFont.truetype(tf, int(base * k))) for t, c, k in TITLE_LINES]
    heights = [int(base * k * 1.14) for _, _, k in TITLE_LINES]   # 行間を詰めて塊にする
    y = (H - sum(heights)) // 2

    # 背後をうっすら暗くして、どんな素材でも文字が立つようにする
    pad = 46
    d.rectangle([0, y - pad, W, y + sum(heights) + pad], fill=(0, 0, 0, 105))

    for (text, col, fnt), lh in zip(fonts, heights):
        x = (W - d.textlength(text, font=fnt)) / 2
        d.text((x, y), text, font=fnt, fill=col, stroke_width=14, stroke_fill="black")
        y += lh
    im.save(out)


def telop(row, font, out):
    from PIL import Image, ImageDraw, ImageFont
    if row["hl"] == "TITLE":
        return draw_title(font, out)
    numcard = row["hl"] == "ALL"
    size = 84 if numcard else 72
    fnt = ImageFont.truetype(font, size)
    im = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)

    def split(line):
        if numcard:
            return [(line, COLORS["red"])]
        hl = row["hl"]
        if hl and hl in line:
            i = line.index(hl)
            segs = []
            if line[:i]:
                segs.append((line[:i], "#FFFFFF"))
            segs.append((hl, COLORS.get(row["hl_color"], "#FFFFFF")))
            if line[i + len(hl):]:
                segs.append((line[i + len(hl):], "#FFFFFF"))
            return segs
        return [(line, "#FFFFFF")]

    lines = [l for l in (row["telop1"], row["telop2"]) if l]
    lh = int(size * 1.28)
    y = int(H * .58) - (len(lines) - 1) * lh // 2
    for line in lines:
        segs = split(line)
        x = (W - sum(d.textlength(t, font=fnt) for t, _ in segs)) / 2
        for txt, col in segs:
            d.text((x, y), txt, font=fnt, fill=col,
                   stroke_width=6 if col != "#FFFFFF" else 5, stroke_fill="black")
            x += d.textlength(txt, font=fnt)
        y += lh
    im.save(out)


def camera(kind, n):
    n = max(n, 2)
    if kind == "zoomout":
        z, x, y = "max(1.08-%f*on,1.0)" % (.08 / n), "iw/2-(iw/zoom/2)", "ih/2-(ih/zoom/2)"
    elif kind == "panleft":
        z, x, y = "1.06", "(iw-iw/zoom)*(1-on/%d)" % n, "ih/2-(ih/zoom/2)"
    elif kind == "panright":
        z, x, y = "1.06", "(iw-iw/zoom)*(on/%d)" % n, "ih/2-(ih/zoom/2)"
    elif kind == "pandown":
        z, x, y = "1.06", "iw/2-(iw/zoom/2)", "(ih-ih/zoom)*(on/%d)" % n
    elif kind == "still":
        z, x, y = "1.0", "iw/2-(iw/zoom/2)", "ih/2-(ih/zoom/2)"
    else:
        z, x, y = "min(1.0+%f*on,1.08)" % (.08 / n), "iw/2-(iw/zoom/2)", "ih/2-(ih/zoom/2)"
    return ("scale=%d:%d:force_original_aspect_ratio=increase,crop=%d:%d,"
            "zoompan=z='%s':d=%d:x='%s':y='%s':s=%dx%d:fps=%d"
            % (W * 2, H * 2, W * 2, H * 2, z, n, x, y, W, H, FPS))


def find_asset(row, cut):
    for d in (IMG, VID):
        if not os.path.isdir(d):
            continue
        hits = sorted(os.path.join(d, f) for f in os.listdir(d)
                      if f.startswith(row["asset"][:3]) or f.startswith("cut%02d_" % cut))
        hits = [h for h in hits if os.path.splitext(h)[1].lower() in VIDEO_EXT | {".png", ".jpg", ".jpeg"}]
        if hits:
            ok = [h for h in hits if "_ok" in os.path.basename(h)]
            return (ok or hits)[0]
    return None


ASSET_ALIAS = {"6": "A06", "7": "A06", "32": "A26"}


def build(font, only_n, out, fixed_dur=None):
    FF = ffmpeg_bin()
    os.makedirs(TMP, exist_ok=True)
    segs = []
    for row in CUTS:
        cut = int(row["cut"])
        if only_n and cut > only_n:
            break
        r = dict(row)
        if row["cut"] in ASSET_ALIAS:
            r["asset"] = ASSET_ALIAS[row["cut"]]
        asset = find_asset(r, cut)
        if asset is None and cut == 0:
            asset = find_asset({"asset": "A01"}, 1)   # タイトルはカット1の絵を借りる
        if asset is None:
            continue
        wav = os.path.join(AUD, "cut%02d.wav" % cut)
        wav = wav if os.path.exists(wav) else None
        if fixed_dur is not None:
            wav = None                      # 音声は最後にまとめて敷く
            dur = fixed_dur.get(cut, 2.2)
        elif r["hl"] == "TITLE":
            dur = wav_seconds(wav) if wav else 3.5
        elif r["hl"] == "ALL":
            dur = NUMCARD_SEC
        elif wav:
            dur = wav_seconds(wav)          # 声の長さがそのままカットの長さになる
        else:
            dur = 2.2
        if fixed_dur is None:
            dur += GAP_ITEM_END if r["item_end"] == "1" else GAP
        if r["hl"] == "TITLE":
            dur = max(dur, 3.2)
        tp = os.path.join(TMP, "t%02d.png" % cut)
        seg = os.path.join(TMP, "s%02d.mp4" % cut)
        telop(r, font, tp)
        frames = int(round(dur * FPS))
        args = [FF, "-hide_banner", "-loglevel", "error", "-y"]
        if os.path.splitext(asset)[1].lower() in VIDEO_EXT:
            args += ["-stream_loop", "-1", "-i", asset]
            vf = ("scale=%d:%d:force_original_aspect_ratio=increase,crop=%d:%d,fps=%d"
                  % (W, H, W, H, FPS))
        else:
            args += ["-loop", "1", "-i", asset]
            vf = camera(r["camera"], frames)
        args += ["-i", tp]
        if wav:
            args += ["-i", wav]
            amap = "[2:a]apad[a]"
        else:
            args += ["-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo"]
            amap = "[2:a]anull[a]"
        args += ["-filter_complex",
                 "[0:v]%s[bg];[bg][1:v]overlay=0:0:format=auto[v];%s" % (vf, amap),
                 "-map", "[v]", "-map", "[a]", "-t", "%.3f" % dur, "-r", str(FPS),
                 "-c:v", "libx264", "-preset", "veryfast", "-crf", "21",
                 "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k",
                 "-ar", "44100", "-ac", "2", seg]
        sh(args)
        segs.append(seg)
        print("   カット%-2d  %s" % (cut, os.path.basename(asset)[:38]))
    if not segs:
        sys.exit("素材が1つも無いので作れませんでした。")
    lst = os.path.join(TMP, "list.txt")
    with open(lst, "w") as f:
        for s in segs:
            f.write("file '%s'\n" % s)
    sh([FF, "-hide_banner", "-loglevel", "error", "-y",
        "-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", out])
    return len(segs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--key", required=True)
    ap.add_argument("--range", type=int, default=0, help="0なら全部")
    ap.add_argument("--out", default="動画.mp4")
    ap.add_argument("--voice", default="", help="VOICEVOXの話者名。空なら声なし")
    ap.add_argument("--vv-host", default="http://127.0.0.1:50021")
    ap.add_argument("--speed", type=float, default=1.10)
    ap.add_argument("--pitch", type=float, default=-0.02)
    ap.add_argument("--intonation", type=float, default=0.90)
    a = ap.parse_args()

    os.makedirs(WORK, exist_ok=True)
    only = a.range or None
    wanted = {c for c in SEARCH if not only or c <= only}

    print("1/4  図版をつくる")
    render_figures()

    voice_file = find_voice_file()
    fixed = None
    if voice_file:
        sec = media_seconds(voice_file)
        print("2/4  用意された音声を使います（%s / %.1f秒）"
              % (os.path.basename(voice_file), sec))
        used = [int(r["cut"]) for r in CUTS if not only or int(r["cut"]) <= only]
        fixed = plan_from_voice(sec, used)
    elif a.voice:
        print("2/4  ナレーションをつくる")
        if not vv_alive(a.vv_host):
            sys.exit("VOICEVOX につながりません。エンジンを起動するか、\n"
                     "自分で作った音声を /content に「声.wav」の名前で置いてください。")
        sp = vv_find_speaker(a.vv_host, a.voice)
        if not sp:
            names = sorted({t[1] for t in vv_speakers(a.vv_host)})
            sys.exit("話者「%s」が見つかりません。使えるのは: %s" % (a.voice, "、".join(names)))
        print("     %s（%s） ID=%d" % (sp[1], sp[2], sp[0]))
        narr_cuts = {c for c in NARRATION if not only or c <= only}
        made, total = make_narration(a.vv_host, sp[0], a.speed, a.pitch, a.intonation, narr_cuts)
        print("     %d本 生成 / 発話 %.1f秒" % (made, total))
    else:
        print("2/4  ナレーションは作りません")

    print("3/4  Pexelsから映像を落とす（%dカット）" % len(wanted))
    got, miss = pexels(a.key, wanted)
    print("     %d本 取得" % got)
    if miss:
        print("     見つからなかったカット: %s" % ", ".join(map(str, miss)))

    font = find_font()
    if not font:
        sys.exit("日本語フォントが見つかりません。")

    print("4/4  動画を組み立てる")
    if fixed is None:
        n = build(font, only, a.out)
    else:
        tmp = os.path.join(WORK, "no_audio.mp4")
        n = build(font, only, tmp, fixed_dur=fixed)
        sh([ffmpeg_bin(), "-hide_banner", "-loglevel", "error", "-y",
            "-i", tmp, "-i", voice_file, "-map", "0:v", "-map", "1:a",
            "-c:v", "copy", "-c:a", "aac", "-b:a", "192k", "-shortest", a.out])
    print("\n完成： %s（%dカット）" % (a.out, n))


if __name__ == "__main__":
    main()
'''
VOICEVOX = r'''
#!/usr/bin/env python3
# Colab上に VOICEVOX ENGINE を落として起動する。
#
# 公式のLinux CPU版リリースを GitHub から取得する。URLは版が上がると変わるので、
# 固定せずにリリース情報を引いて選ぶ。分割書庫（.7z.001, .002 …）にも対応する。
#
#   python3 voicevox_colab.py            # 取得して起動
#   python3 voicevox_colab.py --check    # 起動しているかだけ確認

import argparse
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request

HOST = "http://127.0.0.1:50021"
ROOT = os.path.abspath("voicevox_engine")
RELEASES = "https://api.github.com/repos/VOICEVOX/voicevox_engine/releases/latest"


def alive(timeout=5):
    try:
        urllib.request.urlopen(HOST + "/version", timeout=timeout).read()
        return True
    except Exception:
        return False


def pick_assets(assets):
    # リリースの添付から Linux CPU 版を選ぶ。分割書庫なら全部返す
    def ok(name):
        n = name.lower()
        if "linux" not in n:
            return False
        if any(x in n for x in ("gpu", "nvidia", "cuda", "directml", "macos", "windows")):
            return False
        return "cpu" in n

    hits = [a for a in assets if ok(a["name"])]
    if not hits:
        return []
    # 分割書庫は .7z.001 のように連番。同じ基底名のものをまとめる
    base = sorted(hits, key=lambda a: a["name"])[0]["name"].rsplit(".", 1)[0]
    parts = sorted((a for a in hits if a["name"].startswith(base.rsplit(".7z", 1)[0])),
                   key=lambda a: a["name"])
    return parts or sorted(hits, key=lambda a: a["name"])


def download(url, path):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=1800) as r, open(path, "wb") as f:
        shutil.copyfileobj(r, f, 1 << 20)


def fetch_engine():
    os.makedirs(ROOT, exist_ok=True)
    print("リリース情報を取得しています…")
    req = urllib.request.Request(RELEASES, headers={"User-Agent": "Mozilla/5.0"})
    rel = json.loads(urllib.request.urlopen(req, timeout=60).read())
    parts = pick_assets(rel.get("assets", []))
    if not parts:
        sys.exit("Linux CPU版が見つかりませんでした。\n"
                 "https://github.com/VOICEVOX/voicevox_engine/releases から\n"
                 "linux-cpu のファイルを手で落として、このフォルダに置いてください。")

    print("版 %s / ファイル %d個" % (rel.get("tag_name", "?"), len(parts)))
    local = []
    for i, a in enumerate(parts, 1):
        dst = os.path.join(ROOT, a["name"])
        if os.path.exists(dst) and os.path.getsize(dst) == a.get("size", -1):
            print("  %d/%d %s（取得済み）" % (i, len(parts), a["name"]))
        else:
            print("  %d/%d %s  %.0fMB" % (i, len(parts), a["name"],
                                          a.get("size", 0) / 1e6))
            download(a["browser_download_url"], dst)
        local.append(dst)

    if not shutil.which("7z"):
        subprocess.run("apt-get -qq install -y p7zip-full", shell=True, capture_output=True)

    print("展開しています…")
    first = local[0]
    # 分割書庫は 7z が自分で続きを読む。まずそのまま渡す
    r = subprocess.run(["7z", "x", "-y", "-o" + ROOT, first], capture_output=True, text=True)
    if r.returncode != 0 and len(local) > 1:
        print("  分割のまま展開できなかったので、結合して試します")
        joined = os.path.join(ROOT, "engine.7z")
        with open(joined, "wb") as out:
            for p in local:
                with open(p, "rb") as f:
                    shutil.copyfileobj(f, out, 1 << 20)
        r = subprocess.run(["7z", "x", "-y", "-o" + ROOT, joined],
                           capture_output=True, text=True)
    if r.returncode != 0:
        print("展開に失敗しました:\n" + r.stderr[-800:])
        diagnose()
        sys.exit(1)


def magic(path, n=4):
    try:
        with open(path, "rb") as f:
            return f.read(n)
    except Exception:
        return b""


def find_run():
    # 名前が run のものを全部拾い、実行できる形（ELF）のものを優先する。
    # 中には同名の設定ファイルやスクリプトが混ざることがある
    cands = []
    for root, _, files in os.walk(ROOT):
        for fn in files:
            if fn in ("run", "run.exe"):
                cands.append(os.path.join(root, fn))
    if not cands:
        return None
    elf = [c for c in cands if magic(c) == b"\x7fELF"]
    if elf:
        return max(elf, key=os.path.getsize)
    sh_ = [c for c in cands if magic(c, 2) == b"#!"]
    if sh_:
        return sh_[0]
    return max(cands, key=os.path.getsize)


def diagnose():
    # 失敗したときに、何が展開されたのかを見せる
    print("\n--- 展開されたもの ---")
    if not os.path.isdir(ROOT):
        print("  フォルダがありません:", ROOT)
        return
    total, shown = 0, 0
    for root, _, files in os.walk(ROOT):
        for fn in sorted(files):
            fp = os.path.join(root, fn)
            sz = os.path.getsize(fp)
            total += sz
            if shown < 25:
                rel = os.path.relpath(fp, ROOT)
                print("  %-52s %8.1fMB %s" % (rel[:52], sz / 1e6, magic(fp)[:4]))
                shown += 1
    print("  合計 %.0fMB" % (total / 1e6))
    r = find_run()
    print("  run と判定したもの:", r or "なし")
    if r:
        print("  先頭バイト:", magic(r, 8))


def start():
    run = find_run()
    if not run:
        print("エンジン本体（run）が見つかりません。")
        diagnose()
        return False
    os.chmod(run, 0o755)

    head = magic(run)
    if head == b"\x7fELF":
        cmd = [run, "--host", "127.0.0.1", "--port", "50021"]
    elif magic(run, 2) == b"#!":
        cmd = ["sh", run, "--host", "127.0.0.1", "--port", "50021"]
    else:
        print("run が実行できる形ではありません（先頭 %r）。" % head)
        print("展開が途中で失敗したか、書庫の中身が想定と違います。")
        diagnose()
        return False

    print("起動しています。初回は2〜3分かかります…")
    subprocess.Popen(cmd, cwd=os.path.dirname(run),
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for i in range(90):
        if alive():
            return True
        time.sleep(2)
        if i and i % 15 == 0:
            print("  まだ起動中… (%d秒)" % (i * 2))
    return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--check", action="store_true")
    a = ap.parse_args()

    if alive():
        print("VOICEVOX は起動しています。")
        try:
            names = sorted({s["name"] for s in
                            json.loads(urllib.request.urlopen(HOST + "/speakers", timeout=20).read())})
            print("使える話者: " + "、".join(names))
        except Exception:
            pass
        return
    if a.check:
        sys.exit("VOICEVOX は起動していません。")

    if not find_run():
        fetch_engine()
    if start():
        print("起動しました。③へ進んでください。")
        main()
    else:
        sys.exit("起動できませんでした。もう一度このセルを押すか、"
                 "声なしで作ってから CapCut で足してください。")


if __name__ == "__main__":
    main()
'''
open("pipeline.py", "w", encoding="utf-8").write(PIPELINE)
open("voicevox.py", "w", encoding="utf-8").write(VOICEVOX)

# タイトル用の極太フォント（Dela Gothic One / SIL Open Font License）
import os, urllib.request
os.makedirs("/content/fonts", exist_ok=True)
TITLE_FONT = "/content/fonts/DelaGothicOne-Regular.ttf"
if not os.path.exists(TITLE_FONT):
    for url in ("https://github.com/google/fonts/raw/main/ofl/delagothicone/DelaGothicOne-Regular.ttf",
                "https://raw.githubusercontent.com/google/fonts/main/ofl/delagothicone/DelaGothicOne-Regular.ttf"):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(TITLE_FONT, "wb") as f:
                shutil.copyfileobj(r, f)
            break
        except Exception:
            if os.path.exists(TITLE_FONT):
                os.remove(TITLE_FONT)
if os.path.exists(TITLE_FONT) and os.path.getsize(TITLE_FONT) > 100000:
    print("見出しフォント: Dela Gothic One")
else:
    print("見出しフォントは落とせませんでした。標準の太字で作ります。")

ok = shutil.which("ffmpeg") and glob.glob("/usr/share/fonts/**/NotoSansCJK*", recursive=True)
print("準備できました。②へ。" if ok else "うまくいきませんでした。もう一度押してください。")


In [ ]:
#@title ② 声を用意する { display-mode: "form" }
#@markdown ### 確実なのは、自分で作った音声を置く方法です
#@markdown
#@markdown 1. VOICEVOX の web版を開いて、話者を **青山龍星** にする
#@markdown 2. 台本の「ナレーション全文」を貼って読み上げさせる
#@markdown 3. **1つのファイル**として保存する
#@markdown 4. 左の 📁 から `/content` に **`声.wav`** という名前で入れる
#@markdown
#@markdown 置いてあれば③が勝手に使います。カットの長さもセリフに合わせて割り振ります。
#@markdown
#@markdown ---
#@markdown 下のボタンは、Colab上でVOICEVOXを動かす実験です。
#@markdown **失敗しても問題ありません。**上の方法か、声なしで進めてください。

import subprocess, sys, os
if os.path.exists("/content/声.wav") or os.path.exists("声.wav"):
    print("声.wav が見つかりました。②はこれで完了です。③へ進んでください。")
else:
    p = subprocess.run([sys.executable, "voicevox.py"], capture_output=True, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print(p.stderr[-1200:])
        print("\n起動できませんでした。上の方法で 声.wav を置くか、③で「声を入れない」を選んでください。")


In [ ]:
#@title ③ ▶ 動画をつくる { display-mode: "form" }
#@markdown ### Pexels のキーを貼ってください
Pexelsのキー = ""  #@param {type:"string"}
#@markdown ---
声 = "青山龍星"  #@param ["青山龍星", "声を入れない"]
つくる範囲 = "まず10カットだけ試す"  #@param ["まず10カットだけ試す", "全部つくる"]
#@markdown 話す速さ（1本目は既定のままで）
話速 = 1.1  #@param {type:"slider", min:0.8, max:1.5, step:0.05}

import subprocess, sys
if not Pexelsのキー.strip():
    raise SystemExit("Pexels のキーを貼ってから、もう一度押してください。")

cmd = [sys.executable, "pipeline.py", "--key", Pexelsのキー.strip(),
       "--range", "10" if つくる範囲.startswith("まず") else "0",
       "--speed", str(話速), "--out", "動画.mp4"]
if 声 != "声を入れない":
    cmd += ["--voice", 声]

p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout or "")
if p.returncode != 0:
    print(p.stderr[-2000:])
    raise SystemExit("途中で止まりました。上のメッセージを見てください。")


In [ ]:
#@title ④ 見る・保存する { display-mode: "form" }
from IPython.display import HTML, display
from base64 import b64encode
import os

if not os.path.exists("動画.mp4"):
    raise SystemExit("まだ動画がありません。③を実行してください。")

mb = os.path.getsize("動画.mp4") / 1e6
print("%.1f MB" % mb)
if mb < 40:
    data = b64encode(open("動画.mp4", "rb").read()).decode()
    display(HTML(f'<video width=300 controls src="data:video/mp4;base64,{data}"></video>'))
else:
    print("大きいので、この場では再生せずに保存します。")

from google.colab import files
files.download("動画.mp4")
